### **Rôle principal du fichier :**
Ce script définit un DAG (Directed Acyclic Graph) Apache Airflow nommé scraping_etl_pipeline. Son rôle est d'orchestrer et d'automatiser un pipeline ETL (Extraction, Transformation, Chargement) pour maintenir à jour votre base de données de médicaments. Il est configuré pour s'exécuter de manière hebdomadaire, précisément chaque dimanche à 02h00 UTC. En cas d'échec d'une tâche, il est paramétré pour effectuer jusqu'à 2 tentatives supplémentaires avec un délai de 10 minutes entre chaque.

### **Étapes et flux de tâches (Pipeline) :**

L'orchestration suit un ordre d'exécution strict et séquentiel :

##### **Étape 1 : Extraction des données brutes (scrape_medicaments)**

- Cette tâche utilise un BashOperator pour lancer le script Python scraper_med_ma.py.

- Elle aspire les informations depuis la source (medicament.ma) et stocke le résultat brut dans un fichier CSV situé dans le dossier data/raw/.

##### **Étape 2 : Contrôle qualité et intégrité (verifier_scrape)**

- Cette étape fait appel à un PythonOperator pour exécuter la fonction interne verifier_fichier_scrape.

- Elle agit comme un garde-fou : elle vérifie que le fichier CSV fraîchement créé existe bien et qu'il contient plus d'une ligne (c'est-à-dire au moins une ligne de données en plus de l'en-tête). Si le fichier est vide, le pipeline s'arrête en erreur pour éviter de vider la base de données par la suite.

##### **Étape 3 : Nettoyage et normalisation (transform_data)**

- Une fois les données validées, cette tâche Bash lance transform_med_ma.py.

- Elle transforme le fichier brut en données propres et structurées, générant des fichiers traités (drugs_ma_clean.csv et dci_components.csv) dans le répertoire data/processed/.

##### **Étape 4 : Chargement des médicaments en base (load_drugs)**

- Cette tâche lance load_med_ma.py pour prendre les fichiers CSV nettoyés à l'étape précédente et les insérer ou les mettre à jour dans les tables de votre base de données (drugs_ma et dci_components).

##### **Étape 5 : Chargement des interactions médicamenteuses (load_interactions)**

- La dernière tâche exécute le script load_interactions.py (que nous avons analysé précédemment) pour charger les données du Thésaurus de l'ANSM.

- Bien que le fichier source de l'ANSM ne change pas forcément toutes les semaines, cette étape assure que les éventuels nouveaux médicaments scrappés à l'étape 1 sont bien reliés à leurs interactions potentielles.